In [4]:
import os
import requests
from bs4 import BeautifulSoup
import time

# 1. The NOAA directory URL
noaa_url = "https://www1.ncdc.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
raw_data_dir = "data/raw/"

os.makedirs(raw_data_dir, exist_ok=True)

print("Connecting to NOAA database...")

# 2. Get the webpage content
response = requests.get(noaa_url)
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Find all the links on the page
links = soup.find_all('a')

# 4. Filter for only the "details" files and download them
download_count = 0

for link in links:
    file_name = link.get('href')
    
    # We only want the gzip files that start with "StormEvents_details"
    if file_name and file_name.startswith("StormEvents_details") and file_name.endswith(".csv.gz"):
        file_url = noaa_url + file_name
        file_path = os.path.join(raw_data_dir, file_name)
        
        # Skip downloading if we already have the file
        if not os.path.exists(file_path):
            print(f"Downloading: {file_name}")
            
            # Stream the download to save memory
            with requests.get(file_url, stream=True) as r:
                r.raise_for_status()
                with open(file_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192): 
                        f.write(chunk)
            
            download_count += 1
            # A tiny pause to be polite to NOAA's servers
            time.sleep(0.5) 

print(f"\nSuccessfully downloaded {download_count} new files into {raw_data_dir}.")
print("You are now ready to run the DuckDB 'Master Stitch'!")

Connecting to NOAA database...
Downloading: StormEvents_details-ftp_v1.0_d1950_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1951_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1952_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1953_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1954_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1955_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1956_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1957_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1958_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1959_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1960_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1961_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1962_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1963_c20260316.csv.gz
Downloading: StormEvents_details-ftp_v1.0_d1964_c20260316.c

In [7]:
import duckdb

# 1. Connect to your local database
conn = duckdb.connect('data/processed/tornado_database.duckdb')

# 2. The "Robust Stitch" SQL Query
# We add 'ignore_errors=True' to bypass rows with broken formatting
query = """
CREATE OR REPLACE TABLE ef1_plus_tornadoes AS
SELECT 
    *
FROM 
    read_csv_auto(
        'data/raw/StormEvents_details*.csv.gz', 
        union_by_name=True,
        ignore_errors=True
    )
WHERE 
    EVENT_TYPE = 'Tornado' 
    AND TOR_F_SCALE IN ('F1', 'F2', 'F3', 'F4', 'F5', 'EF1', 'EF2', 'EF3', 'EF4', 'EF5');
"""

print("Executing the Robust Stitch... skipping corrupt rows...")

try:
    # 3. Execute the ETL pipeline
    conn.execute(query)

    # 4. Final verification and count
    result = conn.execute("SELECT COUNT(*) FROM ef1_plus_tornadoes").fetchone()
    print(f"Success! Total EF1+ Tornadoes stitched: {result[0]}")
    
    # Let's peek at the first few rows to make sure it looks right
    preview = conn.execute("SELECT BEGIN_YEARMONTH, STATE, TOR_F_SCALE FROM ef1_plus_tornadoes LIMIT 5").fetchdf()
    print("\nData Preview:")
    print(preview)

except Exception as e:
    print(f"Unexpected error: {e}")

# 5. Close the connection
conn.close()

Executing the Robust Stitch... skipping corrupt rows...
Success! Total EF1+ Tornadoes stitched: 43236

Data Preview:
   BEGIN_YEARMONTH    STATE TOR_F_SCALE
0           195603  ALABAMA          F1
1           195604  ALABAMA          F4
2           195605  ALABAMA          F3
3           195612  ALABAMA          F2
4           195612  ALABAMA          F2
